# 🤖 MedPilot — LLM Server (Qwen2.5-1.5B)

**Chạy vLLM + Qwen thành REST API qua Ngrok**

### Yêu cầu
- Runtime: **GPU (T4 hoặc tốt hơn)**
- Chạy **Runtime → Restart runtime** trước khi chạy nếu gặp lỗi import

### Endpoints sau khi chạy xong
```
POST {ngrok_url}/v1/chat/completions   ← LLM chat
POST {ngrok_url}/v1/extract            ← Medical extraction
GET  {ngrok_url}/health
```

In [ ]:
# ── Cell 1: Cài đặt ───────────────────────────────────────
!pip install vllm transformers fastapi pyngrok uvicorn pydantic -q
print('✅ Cài đặt xong!')

In [ ]:
# ── Cell 2: Kill processes cũ ─────────────────────────────
import subprocess, os
subprocess.run('pkill -f ngrok    || true', shell=True)
subprocess.run('pkill -f uvicorn  || true', shell=True)
subprocess.run('fuser -k 8000/tcp 2>/dev/null || true', shell=True)
os.system('rm -rf ~/.ngrok2/tunnels.yml 2>/dev/null || true')
print('✅ Đã dọn sạch!')

In [ ]:
# ── Cell 3: Load vLLM + Qwen ──────────────────────────────
# ⚠️ Nếu gặp lỗi 'Duplicate registration' → Runtime → Restart runtime → chạy lại
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'

print('🚀 Đang load Qwen2.5-1.5B...')
print('   (Lần đầu cần ~5-10 phút)')

llm = LLM(
    model=MODEL,
    tensor_parallel_size=1,
    max_model_len=4096,          # đủ cho extraction + chat dài
    gpu_memory_utilization=0.85, # dùng tối đa RAM GPU
    dtype='float16'
)
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)

print('✅ vLLM + Qwen loaded!')

In [ ]:
# ── Cell 4: Tạo FastAPI app ───────────────────────────────
import json
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Optional

app = FastAPI(title='MedPilot LLM Server', version='3.0')

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'], allow_methods=['*'], allow_headers=['*']
)


# ─── Schemas ─────────────────────────────────────────────
class Msg(BaseModel):
    role: str
    content: str

class ChatReq(BaseModel):
    model: str = MODEL
    messages: List[Msg]
    temperature: float = 0.7
    max_tokens: int = 2048
    stream: bool = False

class ExtractReq(BaseModel):
    transcript: str
    medical_record: Optional[str] = ''


# ─── Endpoints ───────────────────────────────────────────
@app.get('/')
def root():
    return {'status': 'running', 'model': MODEL,
            'endpoints': ['/v1/chat/completions', '/v1/extract', '/health']}

@app.get('/health')
def health():
    return {'status': 'healthy', 'model': MODEL}


@app.post('/v1/chat/completions')
async def chat_completions(req: ChatReq):
    try:
        msgs = [{'role': m.role, 'content': m.content} for m in req.messages]
        prompt_tokens = tokenizer.apply_chat_template(
            msgs, add_generation_prompt=True, tokenize=True
        )
        params = SamplingParams(
            temperature=req.temperature,
            max_tokens=min(req.max_tokens, 2048),
            stop=['<|im_end|>']
        )
        out = llm.generate([prompt_tokens], params)[0].outputs[0].text
        return {
            'id': 'chatcmpl-medpilot',
            'object': 'chat.completion',
            'model': req.model,
            'choices': [{
                'index': 0,
                'message': {'role': 'assistant', 'content': out},
                'finish_reason': 'stop'
            }],
            'usage': {
                'prompt_tokens': len(prompt_tokens),
                'completion_tokens': len(tokenizer.encode(out)),
                'total_tokens': len(prompt_tokens) + len(tokenizer.encode(out))
            }
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


EXTRACTION_PROMPT = '''Bạn là trợ lý y khoa. Trích xuất thông tin từ hội thoại bác sĩ-bệnh nhân.

TRANSCRIPT: {transcript}
HỒ SƠ CŨ: {medical_record}

Trả lời CHỈ bằng JSON hợp lệ (không giải thích thêm):
{{
  "reason_for_visit": "...",
  "main_symptoms": ["..."],
  "onset_time": "...",
  "lesion_location": "...",
  "related_factors": ["..."],
  "medical_history": ["..."],
  "previous_treatment": ["..."],
  "missing_fields": ["..."],
  "summary": "...",
  "draft_notes": "..."
}}'''

@app.post('/v1/extract')
async def extract_medical_info(req: ExtractReq):
    try:
        prompt = EXTRACTION_PROMPT.format(
            transcript=req.transcript,
            medical_record=req.medical_record or 'Không có'
        )
        msgs = [{'role': 'user', 'content': prompt}]
        prompt_tokens = tokenizer.apply_chat_template(
            msgs, add_generation_prompt=True, tokenize=True
        )
        params = SamplingParams(temperature=0.2, max_tokens=1500)
        out = llm.generate([prompt_tokens], params)[0].outputs[0].text
        return {'extraction': out}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


print('✅ FastAPI app đã định nghĩa!')

In [ ]:
# ── Cell 5: Start server + Ngrok tunnel ──────────────────
import uvicorn, threading, time
from pyngrok import ngrok

# ⚠️ Ngrok auth token
NGROK_TOKEN = '3BM0sCxW0eDMEgAhshblG0Q6cP8_3rdRpdmFj8vkh8oojfZvA'
ngrok.set_auth_token(NGROK_TOKEN)

# Đóng tunnel cũ
for t in ngrok.get_tunnels():
    try: ngrok.disconnect(t.public_url)
    except: pass

# Start server
def run():
    config = uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='warning')
    uvicorn.Server(config).run()

threading.Thread(target=run, daemon=True).start()
time.sleep(3)

# Tạo tunnel
tunnel = ngrok.connect(8000, bind_tls=True)
url = tunnel.public_url

print()
print('=' * 65)
print('🎉  MEDPILOT LLM SERVER SẴN SÀNG!')
print('=' * 65)
print()
print(f'🌐 Ngrok URL : {url}')
print()
print('📋 COPY VÀO FILE .env:')
print('-' * 65)
print(f'VLLM_API_URL={url}/v1/chat/completions')
print(f'EXTRACTION_API_URL={url}/v1/extract')
print('-' * 65)
print()
print('⚠️  Giữ cell này chạy — đóng tab = mất kết nối!')